<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/01.%20API%20Elements%20of%20scikit-learn/ch01_sklearn_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 1: Common Conventions and API Elements of scikit-learn

## Overview

scikit-learn adalah library machine learning Python yang dirancang dengan API yang konsisten dan elegan. Seluruh algoritma mengikuti pola desain yang seragam sehingga mudah digunakan dan dikombinasikan.

### Tiga Abstraksi Utama scikit-learn

| Abstraksi | Interface Utama | Fungsi |
|---|---|---|
| **Estimator** | `fit(X, y)` | Melatih model dari data |
| **Transformer** | `transform(X)`, `fit_transform(X)` | Mengubah/memproses data |
| **Predictor** | `predict(X)`, `predict_proba(X)` | Menghasilkan prediksi |

### Supervised Learning Notation

Given dataset $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^{n}$ where $\mathbf{x}_i \in \mathbb{R}^p$ and $y_i \in \mathbb{R}$ (regression) or $y_i \in \{0,1,...,K\}$ (classification), we seek:

$$\hat{y} = f(\mathbf{x}; \boldsymbol{\theta})$$

where $\boldsymbol{\theta}$ are learned parameters that minimize a loss function $\mathcal{L}$.


In [ ]:
# Import core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print(f"scikit-learn version: {sklearn.__version__}")
print(f"numpy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")


## 1.1 Datasets API

scikit-learn menyediakan berbagai dataset bawaan untuk eksperimen. Dataset dikelompokkan menjadi:
- **Toy datasets**: Dataset kecil untuk belajar (`load_iris`, `load_digits`, `load_diabetes`)
- **Real-world datasets**: Dataset nyata lebih besar (`fetch_california_housing`, `fetch_openml`)
- **Generated datasets**: Dataset sintetis (`make_classification`, `make_regression`, `make_blobs`)


In [ ]:
from sklearn.datasets import (
    load_iris, load_digits, load_diabetes,
    load_wine, load_breast_cancer,
    make_classification, make_regression, make_blobs
)

# Load iris dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

print("=== Iris Dataset ===")
print(f"Features (X) shape: {X_iris.shape}  -> {X_iris.shape[0]} samples, {X_iris.shape[1]} features")
print(f"Target (y) shape:   {y_iris.shape}")
print(f"Feature names:      {iris.feature_names}")
print(f"Target names:       {iris.target_names}")
print(f"Classes:            {np.unique(y_iris)}")

# Create a synthetic classification dataset
X_syn, y_syn = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_redundant=2, random_state=42
)
print(f"\n=== Synthetic Classification ===")
print(f"X shape: {X_syn.shape}, y shape: {y_syn.shape}")
print(f"Class distribution: {np.bincount(y_syn)}")


In [ ]:
# Visualize the iris dataset
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter plot: sepal length vs sepal width
for target, color, label in zip([0,1,2], ['red','green','blue'], iris.target_names):
    mask = y_iris == target
    axes[0].scatter(X_iris[mask, 0], X_iris[mask, 1], c=color, label=label, alpha=0.7, s=50)
axes[0].set_xlabel('Sepal Length (cm)')
axes[0].set_ylabel('Sepal Width (cm)')
axes[0].set_title('Iris: Sepal Length vs Sepal Width')
axes[0].legend()

# Class distribution
axes[1].bar(iris.target_names, np.bincount(y_iris), color=['red','green','blue'], alpha=0.7)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].set_title('Iris Class Distribution')

plt.tight_layout()
plt.show()


## 1.2 Estimator API: fit() dan predict()

Semua model di scikit-learn mengikuti pola:

```python
estimator.fit(X_train, y_train)  # belajar dari data
estimator.predict(X_test)         # prediksi data baru
```

Parameter yang dipelajari disimpan dengan suffix underscore: `estimator.coef_`, `estimator.intercept_`, dll.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

# Instantiate estimator
clf = DecisionTreeClassifier(max_depth=3, random_state=42)

# Fit (learn from training data)
clf.fit(X_train, y_train)

# Predict on test data
y_pred = clf.predict(X_test)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Test accuracy:    {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))


## 1.3 Transformer API: fit_transform()

Transformer mengubah data tanpa menghasilkan label prediksi. Penting: transformer harus di-`fit` hanya pada **training data** untuk menghindari **data leakage**.

$$\text{fit}(X_{train}) \rightarrow \text{simpan parameter} \rightarrow \text{transform}(X_{test})$$


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# BENAR: fit HANYA pada training data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # gunakan parameter dari training

print("=== Before Scaling (Train) ===")
print(f"Mean:  {X_train.mean(axis=0).round(3)}")
print(f"Std:   {X_train.std(axis=0).round(3)}")

print("\n=== After Scaling (Train) ===")
print(f"Mean:  {X_train_scaled.mean(axis=0).round(3)}")
print(f"Std:   {X_train_scaled.std(axis=0).round(3)}")

# Parameters learned by scaler
print(f"\nScaler mean_:  {scaler.mean_.round(3)}")
print(f"Scaler scale_: {scaler.scale_.round(3)}")


## 1.4 Pipeline: Menggabungkan Langkah-langkah

Pipeline menggabungkan preprocessing dan modeling dalam satu objek. Secara matematis, pipeline adalah komposisi fungsi:

$$\hat{y} = f_n\Big(f_{n-1}\big(\ldots f_2(f_1(\mathbf{X}))\big)\Big)$$

Keuntungan Pipeline:
- **Mencegah data leakage**: fit hanya dilakukan pada training data
- **Kode lebih bersih**: semua langkah dalam satu objek
- **Cross-validation yang benar**: CV diterapkan ke seluruh pipeline


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# Build a pipeline: StandardScaler -> SVC
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42))
])

# Train the whole pipeline
pipe.fit(X_train, y_train)

# Predict using pipeline
y_pred_pipe = pipe.predict(X_test)
print(f"Pipeline accuracy: {accuracy_score(y_test, y_pred_pipe):.4f}")

# Access individual steps
print(f"\nPipeline steps: {pipe.steps}")
print(f"Named steps: {list(pipe.named_steps.keys())}")
print(f"Scaler mean_: {pipe.named_steps['scaler'].mean_.round(3)}")


## 1.5 ColumnTransformer: Preprocessing Fitur Heterogen

`ColumnTransformer` memungkinkan penerapan transformer berbeda pada subset kolom yang berbeda (misalnya: scaling untuk numerik, encoding untuk kategorik).


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Create a mixed dataset: numeric + categorical features
np.random.seed(42)
n = 200
data = pd.DataFrame({
    'age': np.random.randint(18, 65, n),
    'income': np.random.exponential(50000, n),
    'education': np.random.choice(['SMA', 'S1', 'S2', 'S3'], n),
    'city': np.random.choice(['Jakarta', 'Bandung', 'Surabaya'], n),
    'target': np.random.randint(0, 2, n)
})

X_mixed = data.drop('target', axis=1)
y_mixed = data['target'].values

# Define which columns are numeric and which are categorical
numeric_features = ['age', 'income']
categorical_features = ['education', 'city']

# Build ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])

# Build full pipeline
full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

X_tr, X_te, y_tr, y_te = train_test_split(X_mixed, y_mixed, test_size=0.3, random_state=42)
full_pipe.fit(X_tr, y_tr)
print(f"Pipeline with ColumnTransformer accuracy: {accuracy_score(y_te, full_pipe.predict(X_te)):.4f}")
print(f"\nInput features: {list(X_mixed.columns)}")

# Get transformed feature names
ohe_features = full_pipe.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)
all_features = numeric_features + list(ohe_features)
print(f"Transformed features: {all_features}")


## 1.6 Cross-Validation

Cross-validation memberikan estimasi performa model yang lebih andal dibanding single train-test split. **K-Fold CV** membagi data menjadi $K$ bagian (folds), melatih pada $K-1$ fold, dan mengevaluasi pada fold yang tersisa:

$$\text{CV Score} = \frac{1}{K} \sum_{k=1}^{K} \text{Score}(M_k, D_{\text{test}}^{(k)})$$


In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold

# Evaluate pipeline with 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# CV on iris with SVM pipeline
scores = cross_val_score(pipe, X_iris, y_iris, cv=cv, scoring='accuracy')

print("=== 5-Fold Stratified CV Results ===")
print(f"Scores per fold: {scores.round(4)}")
print(f"Mean accuracy:   {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")
print(f"95% CI:          [{scores.mean() - 2*scores.std():.4f}, {scores.mean() + 2*scores.std():.4f}]")

# Compare different models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
    'SVM (RBF)': Pipeline([('scaler', StandardScaler()), ('svc', SVC(kernel='rbf'))])
}

print("\n=== Model Comparison (5-Fold CV on Iris) ===")
results = {}
for name, model in models.items():
    sc = cross_val_score(model, X_iris, y_iris, cv=cv, scoring='accuracy')
    results[name] = sc
    print(f"{name:20s}: {sc.mean():.4f} (+/- {sc.std():.4f})")


In [ ]:
# Visualize CV results
fig, ax = plt.subplots(figsize=(9, 5))
names = list(results.keys())
means = [results[n].mean() for n in names]
stds = [results[n].std() for n in names]

bars = ax.bar(names, means, yerr=stds, capsize=6, color=['#2196F3','#4CAF50','#FF9800','#E91E63'], alpha=0.8)
ax.set_xlabel('Model')
ax.set_ylabel('Cross-Validation Accuracy')
ax.set_title('Model Comparison on Iris Dataset (5-Fold CV)')
ax.set_ylim(0.85, 1.02)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.axhline(y=max(means), color='red', linestyle='--', alpha=0.5, label='Best')
ax.legend()
plt.tight_layout()
plt.show()


## 1.7 Feature Importance

Banyak estimator scikit-learn menyediakan atribut `feature_importances_` (tree-based) atau `coef_` (linear models) setelah fitting, yang mengindikasikan kontribusi relatif setiap fitur terhadap prediksi.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importances = rf.feature_importances_
feature_names = iris.feature_names

# Sort features by importance
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(feature_names)))
bars = ax.bar(range(len(feature_names)),
              importances[sorted_idx],
              color=colors, alpha=0.85)
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels([feature_names[i] for i in sorted_idx], rotation=20, ha='right')
ax.set_xlabel('Feature')
ax.set_ylabel('Importance Score')
ax.set_title('Random Forest Feature Importances (Iris Dataset)')
for bar, val in zip(bars, importances[sorted_idx]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

print(f"RF Test Accuracy: {accuracy_score(y_test, rf.predict(X_test)):.4f}")


## 1.8 set_params() dan get_params(): Hyperparameter API

Semua estimator mendukung `get_params()` dan `set_params()` yang memungkinkan akses dan modifikasi hyperparameter secara programatik. Ini adalah fondasi dari `GridSearchCV` dan `RandomizedSearchCV`.


In [ ]:
from sklearn.model_selection import GridSearchCV

# Build a pipeline to tune
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42))
])

# Define hyperparameter grid
# Note: for Pipeline params, use double underscore: 'step_name__param'
param_grid = {
    'svc__C': [0.1, 1.0, 10.0],
    'svc__kernel': ['rbf', 'poly'],
    'svc__gamma': ['scale', 'auto']
}

grid_search = GridSearchCV(
    pipe_svm, param_grid,
    cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_iris, y_iris)

print("=== Grid Search Results ===")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score:   {grid_search.best_score_:.4f}")
print(f"\nBest estimator: {grid_search.best_estimator_}")


## 1.9 Multioutput Estimators

scikit-learn mendukung prediksi multiple output melalui `MultiOutputClassifier` dan `MultiOutputRegressor`. Ini berguna ketika setiap sampel memiliki beberapa label target.


In [ ]:
from sklearn.datasets import make_multilabel_classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression

# Create multi-label dataset
X_ml, y_ml = make_multilabel_classification(
    n_samples=300, n_features=10,
    n_classes=4, n_labels=2,
    random_state=42
)

X_ml_train, X_ml_test, y_ml_train, y_ml_test = train_test_split(
    X_ml, y_ml, test_size=0.3, random_state=42
)

multi_clf = MultiOutputClassifier(
    LogisticRegression(max_iter=500, random_state=42)
)
multi_clf.fit(X_ml_train, y_ml_train)

y_ml_pred = multi_clf.predict(X_ml_test)

print("=== Multi-Label Classification ===")
print(f"X shape: {X_ml.shape}, y shape: {y_ml.shape}")
print(f"Prediction shape: {y_ml_pred.shape}")

# Per-label accuracy
for i in range(y_ml.shape[1]):
    acc = accuracy_score(y_ml_test[:, i], y_ml_pred[:, i])
    print(f"  Label {i} accuracy: {acc:.4f}")


## Ringkasan: Key Takeaways Chapter 1

| Konsep | Poin Kunci |
|---|---|
| **Estimator API** | `fit(X, y)` untuk training; atribut dengan `_` suffix menyimpan learned parameters |
| **Transformer API** | `transform(X)` mengubah data; selalu fit hanya pada training data |
| **Predictor API** | `predict(X)` untuk output deterministik; `predict_proba(X)` untuk probabilitas |
| **Pipeline** | Menggabungkan langkah-langkah secara berurutan; mencegah data leakage |
| **ColumnTransformer** | Terapkan transformer berbeda ke subset kolom berbeda |
| **Cross-Validation** | Estimasi performa model yang lebih andal dengan K-Fold; gunakan StratifiedKFold untuk klasifikasi |
| **Feature Importance** | `feature_importances_` (tree) atau `coef_` (linear) untuk interpretasi model |
| **GridSearchCV** | Optimasi hyperparameter secara sistematis; gunakan double underscore untuk Pipeline params |

### Prinsip Fundamental
1. **Konsistensi**: API yang sama untuk semua estimator memudahkan eksperimen
2. **Composability**: Pipeline dan ColumnTransformer memungkinkan kombinasi modular
3. **No Data Leakage**: Selalu fit transformer hanya pada training data
4. **Reproducibility**: Selalu set `random_state` untuk hasil yang konsisten
